# VaR & CVaR, three ways

_Historical, parametric and Monte Carlo tail risk — then a Kupiec backtest._

**pyportfolios.com** · [/research/var-cvar-three-ways](https://pyportfolios.com/research/var-cvar-three-ways)

> Runs on numpy + pandas + scipy + matplotlib. Synthetic, seeded data — illustrative, not investment advice.

## Three estimators of the same number

In [ ]:
import numpy as np
from scipy.stats import norm, t as student_t

rng = np.random.default_rng(7)
# fat-tailed daily P&L (Student-t, scaled)
pnl = student_t.rvs(5, size=4000, random_state=rng) * 0.01

def hist(pnl, a=0.99):
    loss = -pnl; v = np.quantile(loss, a); return v, loss[loss >= v].mean()

def normal_var(pnl, a=0.99):
    return -(pnl.mean() + pnl.std(ddof=1) * norm.ppf(1 - a))

def t_var(pnl, a=0.99):
    nu, mu, sg = student_t.fit(pnl)
    return -(mu + sg * student_t.ppf(1 - a, nu))

hv, hc = hist(pnl)
print(f"historical    99% VaR {hv:6.2%}   CVaR {hc:6.2%}")
print(f"normal        99% VaR {normal_var(pnl):6.2%}")
print(f"student-t     99% VaR {t_var(pnl):6.2%}")

## Kupiec proportion-of-failures backtest

In [ ]:
def kupiec_pof(exceed, a=0.99):
    n = len(exceed); x = int(exceed.sum()); p = 1 - a
    pi = x / n if n else 0.0
    if x == 0:
        return x, n, np.nan
    lr = -2 * (np.log((1 - p)**(n - x) * p**x)
               - np.log((1 - pi)**(n - x) * pi**x))
    return x, n, lr            # compare LR to chi2(1): 3.84 at 95%

var99 = np.quantile(-pnl, 0.99)
exceed = (-pnl) > var99
x, n, lr = kupiec_pof(exceed)
print(f"exceedances {x}/{n} = {x/n:.2%}  (target 1.00%)   Kupiec LR {lr:.2f}")